# Choose repository files

This standalone tutorial previews files in a local project. It reads names and sizes only; it does not read or execute source code, load a model, or create an index.

On a new computer, first run the setup command in the README. Select QAChat's `.venv` kernel, then run one cell at a time with Shift+Enter.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "qachat/repository.py").is_file():
    root = root.parent
assert (root / "qachat/repository.py").is_file()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
assert Path(sys.prefix).resolve() == (root / ".venv").resolve()

from qachat.repository import scan_repository

print("Python:", sys.executable)

## Choose a local project

Start with QAChat. Replace `root` with another project path if you want. Git is not required. Use a project folder, not your home folder or whole drive.

In [ ]:
REPOSITORY = root
report = scan_repository(REPOSITORY)
print("Repository:", report["repository"])
print("Candidate files:", len(report["selected_files"]))

## Adjust the selection

Choose file extensions, included paths, excluded paths, and a size limit. Empty include paths mean the whole project. Exclusions win. Paths are literal relative paths, not wildcards.

In [ ]:
FILE_TYPES = [".py", ".md"]
INCLUDE_PATHS = []
EXCLUDE_PATHS = ["tests"]
MAX_FILE_BYTES = 1_048_576

In [ ]:
report = scan_repository(
    REPOSITORY,
    extensions=FILE_TYPES,
    include_paths=INCLUDE_PATHS,
    exclude_paths=EXCLUDE_PATHS,
    max_file_bytes=MAX_FILE_BYTES,
)
selected = [
    entry
    for entry in report["entries"]
    if entry["status"] == "selected"
]
for entry in selected[:40]:
    print(entry["path"], entry["size_bytes"])

## Understand skipped entries

Hidden entries, common generated/model folders, weight files, key files, and links are skipped. This selection does not read `.gitignore` and is not a secret scanner.

In [ ]:
skipped = [
    entry
    for entry in report["entries"]
    if entry["status"] == "skipped"
]
for entry in skipped[:40]:
    print(entry["path"], "-", entry["reason"])
assert report["complete"]
assert all(
    0 < entry["size_bytes"] <= MAX_FILE_BYTES
    for entry in selected
)

The result remains in memory. Reading, chunking, embedding, and RAG come in later steps. Clear outputs before committing this tutorial.